# Step 6:  Distributed Processing with Apache Spark (PySpark)

## Objective
This step demonstrates distributed analytics using **Apache Spark** (PySpark) by reading persisted datasets from **MongoDB Atlas**, performing transformations and aggregations in Spark, and writing the results back to MongoDB.

## Environment Notes (Windows)
On Windows, Spark requires:
- `JAVA_HOME` configured (JDK 17)
- Hadoop `winutils.exe` available via `HADOOP_HOME`

These settings are configured below for the current notebook session.


In [15]:
# Step 6 Bootstrap (Windows + MongoDB Atlas + Spark)
import os
from dotenv import load_dotenv

# 1) Load .env
load_dotenv()

# 2) Windows: Java for Spark
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.17.10-hotspot"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

# 3) Windows: Hadoop winutils
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]

# 4) MongoDB connection settings
MONGODB_URI = os.getenv("MONGODB_URI")
MONGODB_DB = os.getenv("MONGODB_DB", "Stocks")

assert MONGODB_URI, "Missing MONGODB_URI in .env"

print("Bootstrap OK")
print("DB:", MONGODB_DB)
print("JAVA_HOME set:", bool(os.environ.get("JAVA_HOME")))
print("HADOOP_HOME set:", bool(os.environ.get("HADOOP_HOME")))


Bootstrap OK
DB: Stocks
JAVA_HOME set: True
HADOOP_HOME set: True


## 6.1 Spark Session Initialization

A Spark session is initialized using PySpark with the MongoDB Spark Connector.  
The connector enables Spark to directly read from and write to MongoDB Atlas collections, allowing distributed analytics over persistently stored financial data.


In [16]:
# Start SparkSession (MongoDB Connector)
from pyspark.sql import SparkSession

MONGO_CONNECTOR = "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0"

spark = (
    SparkSession.builder
    .appName("Step6_Distributed_Analytics")
    .master("local[*]")
    .config("spark.jars.packages", MONGO_CONNECTOR)
    .config("spark.mongodb.read.connection.uri", MONGODB_URI)
    .config("spark.mongodb.write.connection.uri", MONGODB_URI)
    .config("spark.sql.shuffle.partitions", "8")  # small dataset, fewer partitions
    .getOrCreate()
)

print("Spark version:", spark.version)


Spark version: 3.5.3


## 6.2 Reading Persisted Aggregates from MongoDB

The `daily_agg` collection contains daily OHLCV data and rolling statistics computed in Step 5.  
This dataset is loaded into Spark using the MongoDB Spark Connector to enable distributed analytics.

In [26]:
# Step 6.2: Read daily_agg from MongoDB (distributed)

MONGODB_DB = os.getenv("MONGODB_DB", "Stocks")

daily_agg = (
    spark.read.format("mongodb")
    .option("database", MONGODB_DB)
    .option("collection", "daily_agg")
    .load()
)

print("daily_agg rows:", daily_agg.count())
daily_agg.printSchema()


daily_agg rows: 6016
root
 |-- _id: string (nullable = true)
 |-- close: double (nullable = true)
 |-- daily_return: double (nullable = true)
 |-- derived_at: timestamp (nullable = true)
 |-- high: double (nullable = true)
 |-- interval: string (nullable = true)
 |-- log_return: double (nullable = true)
 |-- low: double (nullable = true)
 |-- open: double (nullable = true)
 |-- prev_close: double (nullable = true)
 |-- sma_20: double (nullable = true)
 |-- sma_50: double (nullable = true)
 |-- source: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- vol_20: double (nullable = true)
 |-- volume: integer (nullable = true)



## 6.3 Reading Technical Indicators from MongoDB

The `indicators` collection contains technical indicators (e.g., RSI-14) derived in Step 5.  
This dataset is loaded into Spark to be joined with daily aggregates for downstream analytics.


In [27]:
# Step 6.3: Read indicators from MongoDB (distributed)

indicators = (
    spark.read.format("mongodb")
    .option("database", MONGODB_DB)
    .option("collection", "indicators")
    .load()
)

print("indicators rows:", indicators.count())
indicators.printSchema()


indicators rows: 6016
root
 |-- _id: string (nullable = true)
 |-- avg_gain_14: double (nullable = true)
 |-- avg_loss_14: double (nullable = true)
 |-- derived_at: timestamp (nullable = true)
 |-- interval: string (nullable = true)
 |-- rsi_14: double (nullable = true)
 |-- source: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)



## 6.4 Joining Feature Stores in Spark

The `daily_agg` and `indicators` collections are joined on the natural key `(symbol, timestamp, interval)` to form a unified analytics table.  
This join is executed in Spark to demonstrate distributed processing over persisted data.

In [28]:
# Step 6.4: Distributed join (daily_agg + indicators)
from pyspark.sql import functions as F

joined = (
    daily_agg.alias("d")
    .join(
        indicators.alias("i"),
        on=[
            F.col("d.symbol") == F.col("i.symbol"),
            F.col("d.timestamp") == F.col("i.timestamp"),
            F.col("d.interval") == F.col("i.interval"),
        ],
        how="inner",
    )
    .select(
        F.col("d.symbol").alias("symbol"),
        F.col("d.timestamp").alias("timestamp"),
        F.col("d.interval").alias("interval"),
        F.col("d.close").alias("close"),
        F.col("d.daily_return").alias("daily_return"),
        F.col("d.log_return").alias("log_return"),
        F.col("d.vol_20").alias("vol_20"),
        F.col("d.sma_20").alias("sma_20"),
        F.col("d.sma_50").alias("sma_50"),
        F.col("d.volume").alias("volume"),
        F.col("i.rsi_14").alias("rsi_14"),
    )
).cache()

print("joined rows:", joined.count())

joined rows: 6016


## 6.5 Distributed Analytics: Risk/Return Summary

Using the joined dataset, Spark computes per-symbol summary statistics such as average daily return, volatility, average rolling volatility (vol_20), and average RSI-14.  
Derived annualized metrics are calculated to enable cross-asset comparison. The results are persisted back to MongoDB in a dedicated output collection.


In [29]:
# Step 6.5: Per-symbol risk / return summary (distributed)
from pyspark.sql import functions as F

risk_return = (
    joined
    .groupBy("symbol")
    .agg(
        F.count("*").alias("n_days"),
        F.avg("daily_return").alias("avg_daily_return"),
        F.stddev_samp("daily_return").alias("daily_volatility"),
        F.avg("vol_20").alias("avg_vol_20"),
        F.avg("rsi_14").alias("avg_rsi_14"),
    )
    .withColumn("annualized_return", F.col("avg_daily_return") * F.lit(252.0))
    .withColumn("annualized_volatility", F.col("daily_volatility") * F.sqrt(F.lit(252.0)))
    .withColumn(
        "sharpe_like",
        F.when(
            F.col("annualized_volatility") > 0,
            F.col("annualized_return") / F.col("annualized_volatility")
        ).otherwise(F.lit(None))
    )
)

print("risk_return rows:", risk_return.count())


risk_return rows: 8


## 6.6 Persisting Distributed Analytics Results

The per-symbol risk and return summary computed in Spark is written back to MongoDB Atlas as a persistent output dataset.  
This ensures that analytical results are stored outside the notebook environment and can be reused by downstream visualization or reporting steps.


In [30]:
# Step 6.6: Persist risk/return summary to MongoDB

(
    risk_return
    .write
    .format("mongodb")
    .mode("overwrite")
    .option("database", MONGODB_DB)
    .option("collection", "spark_risk_return")
    .save()
)

print("Wrote collection: spark_risk_return")


Wrote collection: spark_risk_return


## 6.7 Distributed Analytics: Monthly Returns

Spark is used to compute per-symbol monthly returns by identifying the first and last closing prices within each calendar month.  
This demonstrates time-based aggregation and the use of Spark window functions for distributed financial analytics.


In [24]:
# Step 6.7: Monthly returns aggregation (distributed, Spark)

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Create year-month key
dfm = joined.withColumn("year_month", F.date_format(F.col("timestamp"), "yyyy-MM"))

# Windows to get first/last close within each (symbol, year_month)
w_first = Window.partitionBy("symbol", "year_month").orderBy(F.col("timestamp").asc())
w_last  = Window.partitionBy("symbol", "year_month").orderBy(F.col("timestamp").desc())

monthly = (
    dfm
    .withColumn("first_close", F.first("close").over(w_first))
    .withColumn("last_close",  F.first("close").over(w_last))
    .groupBy("symbol", "year_month")
    .agg(
        F.max("first_close").alias("first_close"),
        F.max("last_close").alias("last_close"),
        F.count("*").alias("n_days_in_month"),
    )
    .withColumn(
        "monthly_return",
        (F.col("last_close") / F.col("first_close")) - F.lit(1.0)
    )
    .orderBy("symbol", "year_month")
)

print("monthly rows:", monthly.count())


monthly rows: 296


## 6.8 Persisting Monthly Analytics Results

The monthly return metrics computed using Spark are written back to MongoDB Atlas as a persistent dataset.  
These results serve as an input for visualization and reporting in the final project stage.


In [31]:
# Step 6.8: Persist monthly returns to MongoDB

(
    monthly
    .write
    .format("mongodb")
    .mode("overwrite")
    .option("database", MONGODB_DB)
    .option("collection", "spark_monthly_returns")
    .save()
)

print("Wrote collection: spark_monthly_returns")

Wrote collection: spark_monthly_returns
